In [11]:
from IPython.display import HTML

html_content = """
<!DOCTYPE html>
<html lang="ko">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">

<title>냉장고 체크</title>

<style>

body{
    background:#edf2ef;
    font-family:Arial,sans-serif;
    display:flex;
    justify-content:center;
    padding:20px;
}

.phone{
    width:420px;
    background:white;
    border-radius:30px;
    overflow:hidden;
    box-shadow:0 0 25px rgba(0,0,0,.15);
}

.header{
    background:#4CAF50;
    color:white;
    padding:30px;
}

.header h1{
    margin:0;
    font-size:30px;
}

.header p{
    margin-top:8px;
}

.stats{
    display:flex;
    gap:10px;
    padding:15px;
}

.stat-card{
    flex:1;
    background:white;
    border-radius:20px;
    padding:15px;
    text-align:center;
    box-shadow:0 2px 10px rgba(0,0,0,.08);
}

.stat-card h2{
    margin:0;
    color:#2e7d32;
}

.form-box{
    padding:15px;
    display:flex;
    flex-direction:column;
    gap:10px;
}

.form-box input{
    padding:12px;
    border:1px solid #ccc;
    border-radius:12px;
    font-size:15px;
}

.form-box button{
    background:#4CAF50;
    color:white;
    border:none;
    padding:12px;
    border-radius:12px;
    cursor:pointer;
    font-size:16px;
}

.form-box button:hover{
    background:#388e3c;
}

.alert-box{
    padding:10px 15px;
    font-weight:bold;
    color:#e65100;
}

#foodCards{
    padding:10px;
}

.food-card{
    background:white;
    border-radius:18px;
    padding:15px;
    margin-bottom:12px;
    box-shadow:0 2px 10px rgba(0,0,0,.08);
}

.food-top{
    display:flex;
    justify-content:space-between;
    align-items:center;
}

.food-name{
    font-size:20px;
    font-weight:bold;
}

.food-card p{
    margin:8px 0;
}

.badge{
    color:white;
    padding:6px 12px;
    border-radius:20px;
    font-size:14px;
}

.normal-badge{
    background:#4CAF50;
}

.warning-badge{
    background:#ff9800;
}

.expired-badge{
    background:#f44336;
}

.delete-btn{
    width:100%;
    border:none;
    background:#ef5350;
    color:white;
    padding:10px;
    border-radius:12px;
    margin-top:10px;
    cursor:pointer;
}

.delete-btn:hover{
    background:#d32f2f;
}

.nav{
    display:flex;
    justify-content:space-around;
    padding:15px;
    border-top:1px solid #ddd;
    margin-top:10px;
}

@media(max-width:500px){

    body{
        padding:0;
    }

    .phone{
        width:100%;
        border-radius:0;
    }
}

/* Moved CSS rules */
*{
    color:#111;
}

.header *,
.badge,
.delete-btn{
    color:white !important;
}

</style>
</head>

<body>

<div class="phone">

    <div class="header">
        <h1>🥛 냉장고 체크</h1>
        <p>냉장고 속 식재료를 관리하세요</p>
    </div>

    <div class="stats">

        <div class="stat-card">
            <h2 id="totalCount">0</h2>
            <span>전체 식재료</span>
        </div>

        <div class="stat-card">
            <h2 id="warningCount">0</h2>
            <span>임박 식재료</span>
        </div>

    </div>

    <div class="form-box">

        <input
        type="text"
        id="name"
        placeholder="식재료 이름">

        <input
        type="text"
        id="quantity"
        placeholder="수량">

        <input
        type="date"
        id="expiry">

        <button onclick="addFood()">
            ➕ 식재료 추가
        </button>

    </div>

    <div
    class="alert-box"
    id="alertBox">
    </div>

    <div id="foodCards"></div>

    <div class="nav">
        <div>🏠 홈</div>
        <div>📦 식재료</div>
        <div>⚙ 설정</div>
    </div>

</div>

<script>

let foods =
JSON.parse(
localStorage.getItem("foods")
) || [];

renderFoods();

function getIcon(name){

    if(name.includes("우유")) return "🥛";
    if(name.includes("계란")) return "🥚";
    if(name.includes("사과")) return "🍎";
    if(name.includes("바나나")) return "🍌";
    if(name.includes("토마토")) return "🍅";
    if(name.includes("당근")) return "🥕";
    if(name.includes("고기")) return "🥩";
    if(name.includes("치즈")) return "🧀";
    if(name.includes("라면")) return "🍜";

    return "📦";
}

function addFood(){

    const name =
    document.getElementById("name").value.trim();

    const quantity =
    document.getElementById("quantity").value.trim();

    const expiry =
    document.getElementById("expiry").value;

    if(!name || !quantity || !expiry){

        alert("모든 항목을 입력해주세요.");
        return;
    }

    foods.push({
        name,
        quantity,
        expiry
    });

    saveData();

    document.getElementById("name").value="";
    document.getElementById("quantity").value="";
    document.getElementById("expiry").value="";
}

function saveData(){

    localStorage.setItem(
        "foods",
        JSON.stringify(foods)
    );

    renderFoods();
}

function deleteFood(index){

    if(confirm("삭제하시겠습니까?")){

        foods.splice(index,1);
        saveData();
    }
}

function getDDay(expiryDate){

    const today = new Date();
    const expiry = new Date(expiryDate);

    today.setHours(0,0,0,0);
    expiry.setHours(0,0,0,0);

    return Math.ceil(
        (expiry - today)
        /(1000*60*60*24)
    );
}

function renderFoods(){

    const cards =
    document.getElementById("foodCards");

    const alertBox =
    document.getElementById("alertBox");

    cards.innerHTML="";

    let warningCount=0;

    foods.forEach((food,index)=>{

        const dday =
        getDDay(food.expiry);

        let status="";
        let badge="";
        let ddayText="";

        if(dday < 0){

            status="만료";
            badge="expired-badge";
            ddayText=`D+${Math.abs(dday)}`;

        }else if(dday <=3){

            status="임박";
            badge="warning-badge";
            ddayText=`D-${dday}`;
            warningCount++;

        }else{

            status="정상";
            badge="normal-badge";
            ddayText=`D-${dday}`;
        }

        cards.innerHTML += `

        <div class="food-card">

            <div class="food-top">

                <div>

                    <div class="food-name">
                        ${getIcon(food.name)}
                        ${food.name}
                    </div>

                    <div>
                        ${food.quantity}
                    </div>

                </div>

                <span class="badge ${badge}">
                    ${status}
                </span>

            </div>

            <p>📅 ${food.expiry}</p>

            <p>⏰ ${ddayText}</p>

            <button
            class="delete-btn"
            onclick="deleteFood(${index})">
            🗑 삭제
            </button>

        </div>
        `;
    });

    document.getElementById("totalCount")
    .innerText = foods.length;

    document.getElementById("warningCount")
    .innerText = warningCount;

    if(warningCount>0){

        alertBox.innerHTML =
        `⚠ 임박한 식재료 ${warningCount}개`;

    }else{

        alertBox.innerHTML =
        "✅ 임박한 식재료 없음";
    }
}

</script>

</body>
</html>
"""

HTML(html_content)
